# Wastewater vs cases: sensitivity analysis

Workflow (each step is its own cell, nothing is saved or run inside a function):

1. **Config**: edit `POP_SIZE`, `TEST_DELAY`, `MODEL_TYPE`, `SWEEPS`, etc.
2. **Simulate**: runs Covasim and returns the daily time series (burden, cases, wastewater) for every run. Slow, so do it once per (`POP_SIZE`, `TEST_DELAY`).
3. **Fit**: fits the cases / wastewater / both models on those time series (`"linear"` or `"rank"`). Fast, no re-simulation needed.
4. **Plot**.

To run `test_delay = 0`, `test_delay = 3`, linear and rank: simulate twice (delay 0 and delay 3), then fit each one with `MODEL_TYPE = "linear"` and `"rank"`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

import covasim as cv

Covasim 3.1.8 (2026-05-29) — © 2020-2026 by IDM


In [6]:
from pathlib import Path
print(Path.cwd())                                          # the folder relative paths start from
print(sorted(p.name for p in Path("sensitivity_results").glob("*")) if Path("sensitivity_results").exists() else "no sensitivity_results folder here")

/storage/hcraddock/GitHub/covasim4wastewater/modeling/analysis_sandiego_cases_waste_water
['timeseries_pop10000_delay0.csv']


## Data extraction

In [2]:
def extract_wastewater_regression_data(sim) -> pd.DataFrame:
    """
    Extract daily (burden, cases, wastewater) from a Covasim sim that has been run.

    burden     -> new_infections   (true, unobserved disease burden)
    cases      -> new_diagnoses    (observed via testing/surveillance)
    wastewater -> sum of viral_shedding_hist across agents, per day

    `new_diagnoses` is all zeros unless the sim has a testing intervention.
    """
    results = sim.results

    burden = np.asarray(results['new_infections'].values, dtype=float)

    if 'new_diagnoses' not in results:
        raise KeyError("sim.results has no 'new_diagnoses' — did the sim run?")
    cases = np.asarray(results['new_diagnoses'].values, dtype=float)
    if cases.sum() == 0:
        print("Warning: 'cases' (new_diagnoses) is all zero. "
              "Add a testing intervention, e.g. cv.test_prob(...), to the sim.")

    viral_shedding_hist = np.asarray(results['viral_shedding_hist'])  # (n_people, n_days)
    wastewater = viral_shedding_hist.sum(axis=0)

    n = len(burden)
    return pd.DataFrame({
        'day': np.arange(n),
        'date': [sim.date(d) for d in range(n)],
        'burden': burden,
        'cases': cases,
        'wastewater': wastewater[:n],
    })

## 1. Config
Edit this cell, then run everything below it.

In [ ]:
# ---- Simulation settings (changing these means re-simulating) -------------
POP_SIZE      = 10000
POP_INFECTED  = int(POP_SIZE * 0.0016)   # ~0.16% of the population (16 per 10,000)
N_DAYS        = 120
N_RUNS        = 100                      # random seeds per parameter setting
CHUNK_SIZE    = 10                       # sims held in memory at once (100k-agent sims are big)
RAND_SEED     = 1

TEST_DELAY    = 0                        # <-- fixed testing delay (days) for the symp_prob sweep
BASELINE      = dict(symp_prob=0.03, asymp_prob=0.002, test_delay=TEST_DELAY)
ISO_FACTOR    = dict(h=0.4, s=0.1, w=0.1, c=0.1)   # constant in every simulation

# One-at-a-time sweeps (the swept value overrides BASELINE)
SWEEPS = {
    "symp_prob": [0.01, 0.03, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35],
    # "asymp_prob": [0.0, 0.002, 0.01, 0.05],
    # "test_delay": [1, 3, 7, 10],
}

# ---- Analysis settings (changing these does NOT need re-simulation) --------
MODEL_TYPE       = "linear"                   # "linear" or "rank"
FIZZLE_THRESHOLD = 0.01 * POP_SIZE            # skip runs where <1% of the population was ever infected
BAND             = "iqr"                      # "iqr" or "ci"

# ---- Output locations / names ----------------------------------------------
OUT_DIR  = Path("sensitivity_results")
SIM_TAG  = f"pop{POP_SIZE}_delay{TEST_DELAY}"      # names the simulation files
FIT_TAG  = f"{SIM_TAG}_{MODEL_TYPE}"               # names the fit/plot files
print("SIM_TAG:", SIM_TAG, "| FIT_TAG:", FIT_TAG)

SIM_TAG: pop100000_delay0 | FIT_TAG: pop100000_delay0_linear


## Functions
These only compute and return things. They don't read config globals or write files.

In [4]:
def build_sim(pop_size, pop_infected, n_days, test_kwargs, iso_factor, rand_seed):
    """Build one Covasim sim with a test_prob intervention."""
    return cv.Sim(
        pop_size=pop_size,
        pop_type='hybrid',
        pop_infected=pop_infected,
        location='zambia',
        n_days=n_days,
        n_imports=0,
        interventions=[cv.test_prob(**test_kwargs)],
        iso_factor=iso_factor,
        rand_seed=rand_seed,
        verbose=0,
    )


def simulate_setting(param_name, param_value, baseline, n_runs, chunk_size,
                     pop_size, pop_infected, n_days, iso_factor, rand_seed):
    """Run n_runs stochastic sims at one parameter setting (others at baseline).
    Returns a long DataFrame of daily time series: param, value, run, day, burden, cases, wastewater.
    Sims run in chunks and are discarded after extraction to keep memory bounded."""
    test_kwargs = dict(baseline)
    test_kwargs[param_name] = param_value

    frames = []
    for start in range(0, n_runs, chunk_size):
        k = min(chunk_size, n_runs - start)
        base_sim = build_sim(pop_size, pop_infected, n_days, test_kwargs, iso_factor,
                             rand_seed=rand_seed + start)
        msim = cv.MultiSim(base_sim, n_runs=k)
        msim.run()
        for i, sim in enumerate(msim.sims):
            df = extract_wastewater_regression_data(sim)[['day', 'burden', 'cases', 'wastewater']]
            df.insert(0, 'run', start + i)
            frames.append(df)
        del msim
    out = pd.concat(frames, ignore_index=True)
    out.insert(0, 'value', param_value)
    out.insert(0, 'param', param_name)
    return out


def simulate_sweeps(sweeps, **kwargs):
    """Run simulate_setting for every (param, value) in sweeps; return one concatenated DataFrame."""
    frames = []
    for param_name, values in sweeps.items():
        for value in values:
            print(f"Simulating {param_name} = {value} ...")
            frames.append(simulate_setting(param_name, value, **kwargs))
    return pd.concat(frames, ignore_index=True)


def fit_models(timeseries, model_type="linear", fizzle_threshold=0.0):
    """Fit cases / wastewater / both models on each run's time series.
    model_type 'linear': OLS on the raw values.
    model_type 'rank'  : OLS on the ranks of burden, cases and wastewater (Spearman-style).
    Returns one row per run."""
    if model_type not in ("linear", "rank"):
        raise ValueError("model_type must be 'linear' or 'rank'")

    rows = []
    for (param, value, run), df in timeseries.groupby(['param', 'value', 'run'], sort=False):
        fizzled = df['burden'].sum() < fizzle_threshold
        row = dict(param=param, value=value, run=run, fizzled=fizzled)
        if not fizzled:
            d = df[['burden', 'cases', 'wastewater']]
            if model_type == "rank":
                d = d.rank()
            m_cases = smf.ols('burden ~ cases', data=d).fit()
            m_ww    = smf.ols('burden ~ wastewater', data=d).fit()
            m_both  = smf.ols('burden ~ cases + wastewater', data=d).fit()
            row.update(
                adjr2_cases=m_cases.rsquared_adj,
                adjr2_wastewater=m_ww.rsquared_adj,
                adjr2_both=m_both.rsquared_adj,
                p_add_ww=anova_lm(m_cases, m_both)['Pr(>F)'].iloc[1],     # wastewater adds value beyond cases?
                p_add_cases=anova_lm(m_ww, m_both)['Pr(>F)'].iloc[1],     # cases adds value beyond wastewater?
            )
        rows.append(row)
    return pd.DataFrame(rows)


def summarize(per_run, n_boot=2000, seed=0):
    """Collapse per-run fits to one row per (param, value): median, IQR, bootstrap 95% CI of the median."""
    rng = np.random.default_rng(seed)
    out = []
    for (param, value), g in per_run.groupby(['param', 'value'], sort=False):
        used = g[~g['fizzled']]
        row = dict(param=param, value=value, n_used=len(used), n_fizzled=int(g['fizzled'].sum()))
        for name in ("cases", "wastewater", "both"):
            x = used[f"adjr2_{name}"].to_numpy(dtype=float)
            if len(x):
                boot = np.median(rng.choice(x, size=(n_boot, len(x)), replace=True), axis=1)
                row[f"median_AdjR2_{name}"] = np.median(x)
                row[f"q25_{name}"], row[f"q75_{name}"] = np.percentile(x, [25, 75])
                row[f"ci_low_{name}"], row[f"ci_high_{name}"] = np.percentile(boot, [2.5, 97.5])
            else:
                row.update({f"median_AdjR2_{name}": np.nan, f"q25_{name}": np.nan, f"q75_{name}": np.nan,
                            f"ci_low_{name}": np.nan, f"ci_high_{name}": np.nan})
        row["pct_ww_helps"]    = (used["p_add_ww"] < 0.05).mean() * 100 if len(used) else np.nan
        row["pct_cases_helps"] = (used["p_add_cases"] < 0.05).mean() * 100 if len(used) else np.nan
        out.append(row)
    return pd.DataFrame(out)

## 2. Simulate
The slow step. Only re-run when `POP_SIZE`, `TEST_DELAY`, `N_RUNS`, `SWEEPS` etc. change.

In [ ]:
timeseries = simulate_sweeps(
    SWEEPS,
    baseline=BASELINE, n_runs=N_RUNS, chunk_size=CHUNK_SIZE,
    pop_size=POP_SIZE, pop_infected=POP_INFECTED, n_days=N_DAYS,
    iso_factor=ISO_FACTOR, rand_seed=RAND_SEED,
)
timeseries.head()

### Save simulations

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
timeseries.to_csv(OUT_DIR / f"timeseries_{SIM_TAG}.csv", index=False)
print("Saved", OUT_DIR / f"timeseries_{SIM_TAG}.csv")

### (Optional) Load previously saved simulations instead of re-running
Set `SIM_TAG` in the config cell to the sims you want, then run this.

In [5]:
timeseries = pd.read_csv(OUT_DIR / f"timeseries_{SIM_TAG}.csv")
timeseries.head()

FileNotFoundError: [Errno 2] No such file or directory: 'sensitivity_results/timeseries_pop100000_delay0.csv'

## 3. Fit models
Set `MODEL_TYPE` in the config cell (`"linear"` or `"rank"`) and re-run from here down. No re-simulation.

In [ ]:
per_run = fit_models(timeseries, model_type=MODEL_TYPE, fizzle_threshold=FIZZLE_THRESHOLD)
results = summarize(per_run)
results.round(3)

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
per_run.to_csv(OUT_DIR / f"per_run_{FIT_TAG}.csv", index=False)
results.to_csv(OUT_DIR / f"summary_{FIT_TAG}.csv", index=False)
print("Saved", OUT_DIR / f"summary_{FIT_TAG}.csv")

## 4. Plot

In [ ]:
STYLES = [   # model, legend label, line colour, band colour, line style, marker
    ("cases",      "Cases only",      "black",          "black",          "-",  "o"),
    ("wastewater", "Wastewater only", "cornflowerblue", "cornflowerblue", "-",  "s"),
    ("both",       "Both",            "#08306b",        "#2171b5",        "--", "^"),
]

BAND_LABEL = {"iqr": "Interquartile range across runs", "ci": "95% CI of the median (bootstrap)"}

def draw_adjr2(ax, results, param, band="iqr", fs=14):
    sub = results[results["param"] == param].sort_values("value")
    x = sub["value"].astype(float)
    lo, hi = ("q25", "q75") if band == "iqr" else ("ci_low", "ci_high")
    for name, label, color, band_color, ls, mk in STYLES:
        ax.fill_between(x, sub[f"{lo}_{name}"], sub[f"{hi}_{name}"], color=band_color,
                        alpha=0.20 if name == "both" else 0.15, linewidth=0)
        ax.plot(x, sub[f"median_AdjR2_{name}"], ls=ls, marker=mk, color=color, lw=3, ms=10,
                label=label, clip_on=False)
    ax.set_xlim(left=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(alpha=0.3)
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor="gray", alpha=0.25, label=BAND_LABEL[band]))
    ax.legend(handles=handles, fontsize=fs - 2, loc="lower right", frameon=False)

def plot_symp_prob(results, band="iqr", title=None):
    """Returns the figure; does not save."""
    fig, ax = plt.subplots(figsize=(9.2, 6.9))
    draw_adjr2(ax, results, "symp_prob", band)
    ax.set_xlabel("Symptomatic testing probability (per day)", fontsize=17)
    ax.set_ylabel("Median adjusted R\u00b2 across simulated outbreaks", fontsize=16)
    ax.tick_params(labelsize=14)
    if title:
        ax.set_title(title, fontsize=16)
    fig.tight_layout()
    return fig

In [ ]:
fig = plot_symp_prob(
    results, band=BAND,
    title=f"{MODEL_TYPE} model | pop {POP_SIZE:,} | test delay {TEST_DELAY} d",
)
plt.show()

In [ ]:
fig.savefig(OUT_DIR / f"sensitivity_symp_prob_adjR2_{FIT_TAG}.png", dpi=200)
print("Saved", OUT_DIR / f"sensitivity_symp_prob_adjR2_{FIT_TAG}.png")